# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hapepaAhmed/my-capstone-project/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

# **Read the secret**

In [1]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("Token loaded:", HF_TOKEN is not None)

Token loaded: True


In [2]:
from huggingface_hub import login

login(token=HF_TOKEN)

**inspect the dataset**

In [4]:
from huggingface_hub import list_repo_files

files = list_repo_files(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    token=HF_TOKEN
)

for f in files:
    print(f)

.gitattributes
README.md
dim_clients.parquet
dim_content.parquet
fact_content_daily_performance/month=2025-01/data_0.parquet
fact_content_daily_performance/month=2025-02/data_0.parquet
fact_content_daily_performance/month=2025-03/data_0.parquet
fact_content_daily_performance/month=2025-04/data_0.parquet
fact_content_daily_performance/month=2025-05/data_0.parquet
fact_content_daily_performance/month=2025-06/data_0.parquet
fact_content_daily_performance/month=2025-07/data_0.parquet
fact_content_daily_performance/month=2025-08/data_0.parquet
fact_content_daily_performance/month=2025-09/data_0.parquet
fact_content_daily_performance/month=2025-10/data_0.parquet
fact_content_daily_performance/month=2025-11/data_0.parquet
fact_content_daily_performance/month=2025-12/data_0.parquet
fact_content_daily_performance/month=2026-01/data_0.parquet
fact_content_daily_performance/month=2026-02/data_0.parquet
fact_content_daily_performance/month=2026-03/data_0.parquet
fact_content_daily_performance/mont

# Download the March 2026 Parquet file

In [5]:
from huggingface_hub import hf_hub_download

parquet_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    token=HF_TOKEN,
)

print(parquet_path)

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-03/data_0.parquet


In [6]:
import pandas as pd

df = pd.read_parquet(parquet_path)
df.head()

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,None,20,0,67,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,None,1,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,None,125,1,616,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,None,7,0,28,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,None,11,0,25,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### Unit of Analysis + Time Window

- **One row:** One content item (`content_hash_id`) for one client (`client_hash_id`) on one reporting date (`report_date`).
- **Table(s):** I use the `fact_content_daily_performance` table because it contains the daily search and analytics performance metrics needed for ranking analysis.
- **Time window:** March 2026 (`month=2026-03`), following the assignment recommendation to use a mid-panel month instead of the final month.
- **Purpose:** Each row contains performance signals such as impressions, clicks, average position, sessions, and engagement metrics that can be used to analyze and prioritize content for optimization.

In [11]:
#data base shape
print("Rows, Columns:",df.shape)

#list of all columns
print(df.columns.tolist())

#(one row = one client + one content item + one date)
duplicates = df.duplicated(
    subset=["report_date", "client_hash_id", "content_hash_id"]
).sum()

print("Duplicate rows for the grain:", duplicates)

#Verify the date span
print("Start date:", df["report_date"].min())
print("End date:", df["report_date"].max())

Rows, Columns: (9841378, 30)
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events']
Duplicate rows for the grain: 0
Start date: 2026-03-01
End date: 2026-03-31


In [12]:
df["report_date"] = pd.to_datetime(df["report_date"])

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Fields: Feature / Label / Context / Excluded

#### Feature Fields
These fields will be used as input features for ranking content pages:

- `gsc_impressions`
- `gsc_clicks`
- `gsc_avg_position`
- `ga4_sessions`
- `scroll_events`

These features describe the search visibility, traffic, and engagement of each content item.

#### Label / Proxy
The dataset does not contain a direct optimization-priority label. Therefore, I will use an **optimization priority score** as a proxy target. This score represents how important it is to optimize a content page based on its ranking signals.

#### Context Fields
These fields provide context and uniquely identify each record but are not used as predictive features:

- `report_date`
- `client_hash_id`
- `content_hash_id`

#### Excluded Fields
The following fields are excluded:

- `sessions_ai`
- `ai_chatgpt`
- `ai_perplexity`
- `ai_gemini`
- `ai_copilot`
- `ai_claude`
- `ai_meta`
- `ai_other`

**Why excluded?**

These fields measure AI referral traffic rather than traditional search ranking performance. Since my project focuses on **Ranking Signal Analysis**, I will concentrate on Google Search Console and GA4 metrics.

In [14]:
# Organize fields into the four buckets

features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "scroll_events"
]

label = [
    "optimization_priority_score (proxy)"
]

context = [
    "report_date",
    "client_hash_id",
    "content_hash_id"
]

excluded = [
    "sessions_ai",
    "ai_chatgpt",
    "ai_perplexity",
    "ai_gemini",
    "ai_copilot",
    "ai_claude",
    "ai_meta",
    "ai_other"
]

print("=== FEATURES ===")
print(features)

print("\n=== LABEL / PROXY ===")
print(label)

print("\n=== CONTEXT ===")
print(context)

print("\n=== EXCLUDED ===")
print(excluded)

=== FEATURES ===
['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_sessions', 'scroll_events']

=== LABEL / PROXY ===
['optimization_priority_score (proxy)']

=== CONTEXT ===
['report_date', 'client_hash_id', 'content_hash_id']

=== EXCLUDED ===
['sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other']


In [15]:
import pandas as pd

field_groups = pd.DataFrame({
    "Feature": pd.Series(features),
    "Label/Proxy": pd.Series(label),
    "Context": pd.Series(context),
    "Excluded": pd.Series(excluded)
})

field_groups


,Feature,Label/Proxy,Context,Excluded
0,gsc_impressions,optimization_priority_score (proxy),report_date,sessions_ai
1,gsc_clicks,NaN,client_hash_id,ai_chatgpt
2,gsc_avg_position,NaN,content_hash_id,ai_perplexity
3,ga4_sessions,NaN,NaN,ai_gemini
4,scroll_events,NaN,NaN,ai_copilot
5,NaN,NaN,NaN,ai_claude
6,NaN,NaN,NaN,ai_meta
7,NaN,NaN,NaN,ai_other


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Verify the Data Contract

The following queries verify the main assumptions of my data contract.

1. **Grain:** Confirm that each row represents one content item for one client on one reporting date by checking for duplicate combinations of `report_date`, `client_hash_id`, and `content_hash_id`.

2. **Counts and Time Window:** Verify the dataset size and confirm that all records belong to the selected analysis period (March 2026).

3. **Missing Values:** Check the selected feature columns for missing values to understand the completeness of the data before building features or models.

4. **Availability:** Verify that Google Search Console and GA4 data are available by filtering rows where `gsc_data_available` and `ga4_data_available` are `True`.

In [20]:
duplicates = df.duplicated(
    subset=["report_date", "client_hash_id", "content_hash_id"]
).sum()

print("Duplicate rows:", duplicates)


missing = pd.DataFrame({
    "Missing Count": df[features].isnull().sum(),
    "Missing %": (df[features].isnull().mean() * 100).round(2)
})

print(missing)



print("Dataset shape:", df.shape)

print("Start date:", df["report_date"].min())
print("End date:", df["report_date"].max())


available = df[
    (df["gsc_data_available"] == True) &
    (df["ga4_data_available"] == True)
]

print("Rows with both GSC and GA4 data available:", len(available))


Duplicate rows: 0
                  Missing Count  Missing %
gsc_impressions               0       0.00
gsc_clicks                    0       0.00
gsc_avg_position        6230317      63.31
ga4_sessions            3018741      30.67
scroll_events           3018741      30.67
Dataset shape: (9841378, 30)
Start date: 2026-03-01 00:00:00
End date: 2026-03-31 00:00:00
Rows with both GSC and GA4 data available: 364347


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**ANswer:**

          This dataset has several limitations:

          - It only includes historical performance data, so it cannot explain why a page's ranking changed (for example, due to algorithm updates or competitor actions).
          - Some rows may contain only Google Search Console (GSC) data or only GA4 data, which can result in incomplete performance information.
          - I am using only the March 2026 data, so the analysis does not capture long-term trends or seasonal patterns.
          - The dataset does not include a true optimization-priority label, so a proxy target must be defined for modeling.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.